# Batch Correction Evaluation Pipeline
Evaluates pyCombat and NormAE on MALDI MSI prostate cancer data.


In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import logging
logging.basicConfig(level=logging.INFO)


## 1. Load or Create Synthetic Data


In [ ]:
from src.data.data_loader import get_batch_info, get_normalization_names

NORM_NAMES = get_normalization_names()
BATCH_INFO = get_batch_info()

# Synthetic example
n_samples = 500
n_features = 159
n_batches = 7
n_tissues = 4
rng = np.random.RandomState(42)

# Create batch and tissue labels
batch_labels = np.repeat(np.arange(n_batches), [80, 10, 60, 80, 100, 90, 80])[:n_samples]
np.random.shuffle(batch_labels)
tissue_labels = rng.randint(0, n_tissues, size=n_samples)

# Create synthetic data with batch effects
data_raw = rng.randn(n_samples, n_features)
for b in range(n_batches):
    data_raw[batch_labels == b] += rng.randn(n_features) * 2  # batch effect

df_raw = pd.DataFrame(data_raw, columns=[f'mz_{i}' for i in range(n_features)])

data_dict_raw = {norm: df_raw.copy() for norm in NORM_NAMES}
print(f'Synthetic data: {n_samples} samples, {n_features} features, {n_batches} batches')


## 2. Apply pyCombat Correction


In [ ]:
try:
    from src.batch_correction.combat_correction import apply_pycombat_per_normalization
    data_dict_pycombat = apply_pycombat_per_normalization(data_dict_raw, batch_labels)
    print('pyCombat correction completed.')
except Exception as e:
    print(f'pyCombat not available: {e}. Using raw data.')
    data_dict_pycombat = data_dict_raw.copy()


## 3. Apply NormAE Correction


In [ ]:
from src.batch_correction.normae_correction import apply_normae_per_normalization
data_dict_normae = apply_normae_per_normalization(
    data_dict_raw, batch_labels, n_epochs=20
)
print('NormAE correction completed.')


## 4. Run Full Evaluation


In [ ]:
from src.evaluation.evaluate_correction import run_full_evaluation, get_best_method, print_results_table

# Build nested dict
full_dict = {}
for norm in NORM_NAMES:
    full_dict[norm] = {
        'raw': data_dict_raw[norm],
        'pycombat': data_dict_pycombat[norm],
        'normae': data_dict_normae[norm],
    }

results_df = run_full_evaluation(full_dict, batch_labels, tissue_labels, sample_size=500)
print(f'Evaluation complete: {len(results_df)} rows')


## 5. Display Results Table


In [ ]:
print_results_table(results_df)


## 6. PCA Plots


In [ ]:
from src.visualization.plot_batch_effects import plot_pca_before_after
norm = NORM_NAMES[0]
plot_pca_before_after(
    data_dict_raw[norm].values,
    data_dict_normae[norm].values,
    batch_labels, tissue_labels,
    'NormAE', '/tmp/figures'
)
from IPython.display import Image
Image('/tmp/figures/pca_before_after_NormAE.png')


## 7. Metrics Heatmap


In [ ]:
from src.visualization.plot_batch_effects import plot_metrics_heatmap
plot_metrics_heatmap(results_df, '/tmp/figures')
Image('/tmp/figures/metrics_heatmap.png')


## 8. Best Method Recommendation


In [ ]:
best = get_best_method(results_df)
print('Best method:')
print(best[['method', 'normalization', 'bio_batch_f1', 'knn_accuracy', 'silhouette_batch']])
